# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Station

In [12]:
snap = station.snapshot(update=True) # <- updates parameters in station

In [13]:
verticalsnap = snap['instruments']['mso5']['submodules']['channels']['channels']['mso5_ch1']['parameters'].keys()

In [14]:
horizontalsnap = snap['instruments']['mso5']['parameters'].keys()

# Import

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260412-30700-qcodes.log
Experiment loaded. Last ID no: 464


In [3]:
station = Station(config_file="friesland.yaml")

In [3]:
dmm = station.load_instrument("dmm", revive_instance=True)

Connected to: Agilent Technologies 34410A (serial:MY47027892, firmware:2.35-2.35-0.09-46-09) in 0.11s


In [4]:
yoko = station.load_instrument("yoko", revive_instance=True)

Connected to: YOKOGAWA GS210 (serial:91T928105, firmware:2.02) in 0.72s


In [5]:
laser = station.load_instrument("laser", revive_instance=True)

2026-04-10 10:27:07,746 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ D:\SNSPD\SNSPD2\PPCL550.py:4: QCoDeSDeprecationWarning: The `qcodes.utils.helpers` module is deprecated. Please consult the api documentation at https://microsoft.github.io/Qcodes/api/index.html for alternatives.
  from qcodes.utils.helpers import create_on_off_val_mapping



Connected to: PurePhotonic PPCL550 (serial:PP70AJ005, firmware:PV 2.0.0:HW 3.0.0:FW 7.0.0:AS C1:OT 1.0.0) in 1.55s


In [11]:
MS = station.load_instrument("mso5", revive_instance=True)

In [12]:
pm100usb = station.load_instrument("pm100usb", revive_instance=True)

In [13]:
pms120 = station.load_instrument("pms120", revive_instance=True)

In [14]:
tc = station.load_instrument("fridge", revive_instance=True)

In [15]:
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

# Parameters

In [28]:
device_name_1 = 'Line 1 R7C6'
device_name_2 = 'Line 2 Old Device'
att_screw = 'VOA50PM' 
att_blue = 'V1550PA'
# Beam splitter ratio measured ...? 
bs90 = 0.9118079062146573 
bs10 = 0.08819209378534265
h_time = 3e-6 # time for trace acquisition
h_pos = 10
v_scale=150e-3
# v_trigger = 600e-3 # set trigger well above 
v_trigger = -200e-3

# Attenuator voltages for V1550PA from calibration
ID = 242
data = load_by_id(ID).get_parameter_data()
v_attenuator = data['v_attenuator']['v_attenuator']

# Trace Captures

With yoko set to 10uA but not responding 

In [23]:
osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)

if MS.channels[0].clipping(): 
    print('Error: Clipping')

print(yoko.current())

############################ TURN LASER ON ############################ 
laser.enable(True)

time.sleep(10)
print(f'Laser enable status: {laser.enable()}')

for v in np.arange(3.4, 5.05, 0.1)[::-1]:

    p_att.write(f'VOLT {v}')

    print(f'Voltage on attenuator is {p_att.ask('VOLT?')}')

    for i in range(3):

        # capture the trace 
        capture_trace_simple(MS, dmm, v_attenuator = v, station=station)

        time.sleep(5)
    
    

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

1e-05
Laser enable status: True
Voltage on attenuator is 5


2026-04-10 00:08:49,920 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 265. 
265
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


2026-04-10 00:09:12,223 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 266. 
266
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


...

Starting experimental run with id: 313. 
313
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


2026-04-10 00:24:13,641 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 314. 
314
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


2026-04-10 00:24:29,866 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 315. 
315
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Laser enable status: False


In [49]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False


Dialled voltage on attenuator up from 3.5 to 5 in between saw screen stay full of counts on 5V then suddenly went away. 

# 10/4/2026

When I put the trigger at -200mV, there is constantly a trace on the screen (12uA bias, laser off). When I put the trigger at 200mV, peaks are rare. 

Yoko connection is breaking.

In [17]:
capture_trace(MS, dmm, p_att)

Starting experimental run with id: 321. 
321
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;1.120E-9;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


In [ ]:
capture_trace(MS, dmm, p_att, station=True)

In [18]:
osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)

Trigger set to 600

In [20]:
print(v_scale) 
print(h_time)
print(h_pos)
print(v_trigger)

osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)

if MS.channels[0].clipping(): 
    print('Error: Clipping')


############################ TURN LASER ON ############################ 
laser.enable(True)

time.sleep(10)
print(f'Laser enable status: {laser.enable()}')


for v in np.arange(3.4, 5.05, 0.1)[::-1]:

    # Set attenuator voltage 
    p_att.write(f'VOLT {v}')

    time.sleep(20) # to let the trace change because of that weird thing that happens with the trace not adjusting immediately

    print(f'Voltage on attenuator is {p_att.ask('VOLT?')}')

    for i in range(3):

        # capture the trace 
        capture_trace(MS, dmm, p_att, station=station)

        time.sleep(5)
    
    

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

0.15
3e-06
10
Laser enable status: True
Voltage on attenuator is 5


2026-04-10 10:30:39,146 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 325. 
325
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


2026-04-10 10:30:58,399 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 326. 
326
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


...

2026-04-10 10:32:16,436 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 329. 
329
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


2026-04-10 10:32:38,585 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 330. 
330
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


KeyboardInterrupt: 

In [22]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False


Aborted - better to have the trigger below otherwise we won't see it

In [31]:
np.arange(3.4, 6.1, 0.1)[::-1]

array([6. , 5.9, 5.8, 5.7, 5.6, 5.5, 5.4, 5.3, 5.2, 5.1, 5. , 4.9, 4.8,
       4.7, 4.6, 4.5, 4.4, 4.3, 4.2, 4.1, 4. , 3.9, 3.8, 3.7, 3.6, 3.5,
       3.4])

In [34]:
print(v_scale) 
print(h_time)
print(h_pos)
print(v_trigger)

0.15
3e-06
10
-0.2


In [32]:
print(v_scale) 
print(h_time)
print(h_pos)
print(v_trigger)

osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)

if MS.channels[0].clipping(): 
    print('Error: Clipping')


############################ TURN LASER ON ############################ 
laser.enable(True)

time.sleep(10)
print(f'Laser enable status: {laser.enable()}')


for v in np.arange(3.4, 6.1, 0.1)[::-1]:

    # Set attenuator voltage 
    p_att.write(f'VOLT {v}')

    time.sleep(20) # to let the trace change because of that weird thing that happens with the trace not adjusting immediately

    print(f'Voltage on attenuator is {p_att.ask('VOLT?')}')

    for i in range(2):

        # capture the trace 
        capture_trace(MS, dmm, p_att, station=station)

        time.sleep(5)
    
    
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

0.15
3e-06
10
-0.2
Laser enable status: True
Voltage on attenuator is 6


2026-04-10 10:40:35,732 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 331. 
331
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;1.120E-9;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


2026-04-10 10:41:01,002 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 332. 
332
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;320.00E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Voltage on attenuator is 5.9


...

Starting experimental run with id: 382. 
382
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;320.00E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Voltage on attenuator is 3.4


2026-04-10 11:09:57,676 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 383. 
383
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;160.00E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


2026-04-10 11:10:20,093 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 384. 
384
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;1.60E-9;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Laser enable status: False


Running again with proper shielding (box closed, fibre wrapped)

In [35]:
print(v_scale) 
print(h_time)
print(h_pos)
print(v_trigger)

osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)

if MS.channels[0].clipping(): 
    print('Error: Clipping')


############################ TURN LASER ON ############################ 
laser.enable(True)

time.sleep(10)
print(f'Laser enable status: {laser.enable()}')


for v in np.arange(3.4, 6.1, 0.1)[::-1]:

    # Set attenuator voltage 
    p_att.write(f'VOLT {v}')

    time.sleep(20) # to let the trace change because of that weird thing that happens with the trace not adjusting immediately

    print(f'Voltage on attenuator is {p_att.ask('VOLT?')}')

    for i in range(2):

        # capture the trace 
        capture_trace(MS, dmm, p_att, station=station)

        time.sleep(5)
    
    
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

0.15
3e-06
10
-0.2
Laser enable status: True
Voltage on attenuator is 6


2026-04-10 11:52:20,851 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()

2026-04-10 11:52:36,740 ¦ asyncio ¦ ERROR ¦ base_events ¦ default_exception_handler ¦ 1879 ¦ Task was destroyed but it is pending!
task: <Task pending name='Task-238' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-239' coro=<Kernel.shell_main() running at C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
2026-04-10 11:52:36,742 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\env

Starting experimental run with id: 385. 
385
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;800.0E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


2026-04-10 11:52:43,063 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 386. 
386
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;640.00E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Voltage on attenuator is 5.9


...

Starting experimental run with id: 436. 
436
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;395.000E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Voltage on attenuator is 3.4


2026-04-10 12:21:22,212 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 437. 
437
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;320.00E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1


2026-04-10 12:21:44,691 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 438. 
438
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 38.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;640.00E-12;188;"V";6.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Laser enable status: False
